In [ ]:
# ============================================================
# DataBrief AI — Ato 3
# Célula 1: Setup — rodar sempre ao iniciar a sessão
# ============================================================
import subprocess, sys, os

# ── Modo de execução desta sessão ──────────────────────────
# Troque manualmente antes de rodar: "GPU_LOCAL", "CPU_LOCAL" ou "OPENAI".
# Nunca é adivinhado/trocado sozinho depois — decide aqui, uma vez, pra
# sessão inteira (ver graph/agent.py: obter_modo_ativo()).
MODO_EXECUCAO = "GPU_LOCAL"

if MODO_EXECUCAO not in ("GPU_LOCAL", "CPU_LOCAL", "OPENAI"):
    raise ValueError(
        f"MODO_EXECUCAO='{MODO_EXECUCAO}' inválido — use 'GPU_LOCAL', "
        "'CPU_LOCAL' ou 'OPENAI'."
    )

# Garante pasta limpa — evita conflitos de clone duplicado
os.chdir("/content")
subprocess.run(["rm", "-rf", "databrief-ai"], check=True)
subprocess.run(["git", "clone",
                "https://github.com/philsimoes/databrief-ai.git"], check=True)
os.chdir("databrief-ai")
sys.path.insert(0, ".")

# Instalar dependências
subprocess.run([
    "pip", "install", "-q",
    "pydantic==2.11.4",
    "langgraph==0.2.73",
    "langchain-core==0.3.62",
    "langchain==0.3.25",  # graph/agent.py faz "import langchain" direto
    "piper-tts",
    "transformers==4.51.3",
    "accelerate==1.6.0",
    "bitsandbytes>=0.46.1",   # só usado em GPU_LOCAL — instalar sempre não
                              # atrapalha CPU_LOCAL/OPENAI, só fica sem uso
    "openai==1.82.0",         # <-- novo: chamar_openai() em graph/agent.py
                              # faz "from openai import OpenAI" — só é
                              # importado de fato quando MODO_EXECUCAO=
                              # "OPENAI", mas precisa estar instalado pra
                              # não quebrar com ModuleNotFoundError se você
                              # trocar o modo
    "librosa",
    "soundfile",
    "pdfplumber",
    "python-docx",
    "gradio==5.29.1",
], check=True)

# Verificar GPU
try:
    result = subprocess.run(["nvidia-smi", "--query-gpu=name,memory.total",
                             "--format=csv,noheader"], capture_output=True, text=True)
    gpu_disponivel = (result.returncode == 0)
except FileNotFoundError:
    gpu_disponivel = False

if gpu_disponivel:
    print(f"✅ GPU: {result.stdout.strip()}")
else:
    print("⚠️  Sem GPU disponível nesta sessão do Colab.")

# Trava cedo e com erro claro — evita o erro genérico de bitsandbytes que
# você viu, quando MODO_EXECUCAO="GPU_LOCAL" mas a sessão não tem GPU.
if MODO_EXECUCAO == "GPU_LOCAL" and not gpu_disponivel:
    raise RuntimeError(
        "MODO_EXECUCAO='GPU_LOCAL' mas esta sessão do Colab não tem GPU "
        "conectada. Troque para MODO_EXECUCAO='CPU_LOCAL' nesta célula, ou "
        "reconecte o runtime com GPU (Ambiente de execução > Alterar tipo "
        "de ambiente de execução > GPU) e rode a Célula 1 de novo."
    )

print(f"✅ Setup completo — modo: {MODO_EXECUCAO}")

In [ ]:
# ============================================================
# Célula 2: Carregar o modelo de linguagem conforme MODO_EXECUCAO
#   GPU_LOCAL -> Qwen3-4B, 4-bit (bitsandbytes, exige CUDA)
#   CPU_LOCAL -> Qwen3-1.7B, bf16/fp16 puro (sem quantização)
#   OPENAI    -> nenhum Qwen local; só valida a chave da API
# ============================================================
import time
import torch
from schemas.models import ModoExecucao

_MODO_MAP = {
    "GPU_LOCAL": ModoExecucao.GPU_LOCAL,
    "CPU_LOCAL": ModoExecucao.CPU_LOCAL,
    "OPENAI":    ModoExecucao.OPENAI,
}
modo_execucao_enum = _MODO_MAP[MODO_EXECUCAO]

model = None
tokenizer = None

if MODO_EXECUCAO in ("GPU_LOCAL", "CPU_LOCAL"):
    from transformers import AutoTokenizer, AutoModelForCausalLM
    from google.colab import userdata
    from huggingface_hub import login

    login(token=userdata.get("HF_TOKEN"))

    if MODO_EXECUCAO == "GPU_LOCAL":
        MODELO = "Qwen/Qwen3-4B"
        print(f"⏳ Carregando {MODELO} em 4-bit (GPU_LOCAL)...")
        print("   Primeira vez: ~4.5 GB para baixar. Aguarde.\n")

        from transformers import BitsAndBytesConfig
        config_4bit = BitsAndBytesConfig(
            load_in_4bit=True,
            bnb_4bit_quant_type="nf4",
            bnb_4bit_compute_dtype=torch.float16,
            bnb_4bit_use_double_quant=True,
        )

        t0 = time.time()
        tokenizer = AutoTokenizer.from_pretrained(MODELO)
        model = AutoModelForCausalLM.from_pretrained(
            MODELO,
            quantization_config=config_4bit,
            device_map="auto",
        )
        tempo = time.time() - t0
        vram = torch.cuda.memory_allocated() / 1024**3
        print(f"✅ Qwen3-4B carregado em {tempo:.0f}s")
        print(f"   VRAM usada: {vram:.1f} GB")

    else:  # CPU_LOCAL
        MODELO = "Qwen/Qwen3-1.7B"
        # bf16 puro, sem bitsandbytes — bitsandbytes 4-bit exige CUDA (ou
        # backends específicos que o Colab free não tem), não roda em CPU
        # comum. device_map="auto" deixa usar GPU se por acaso existir uma
        # disponível nesta sessão, mas CPU_LOCAL funciona igual sem.
        tipo_dado = torch.float16 if torch.cuda.is_available() else torch.bfloat16
        print(f"⏳ Carregando {MODELO} em {tipo_dado} (CPU_LOCAL, sem quantização)...")
        print("   Primeira vez: ~3.5 GB para baixar. Aguarde.\n")

        t0 = time.time()
        tokenizer = AutoTokenizer.from_pretrained(MODELO)
        model = AutoModelForCausalLM.from_pretrained(
            MODELO,
            torch_dtype=tipo_dado,
            device_map="auto",
        )
        tempo = time.time() - t0
        print(f"✅ Qwen3-1.7B carregado em {tempo:.0f}s")

else:  # OPENAI
    from google.colab import userdata
    import os as _os

    chave = userdata.get("OPENAI_API_KEY")
    if not chave:
        raise RuntimeError(
            "MODO_EXECUCAO='OPENAI' mas não existe um secret 'OPENAI_API_KEY' "
            "configurado no Colab. Adicione em: ícone de chave 🔑 na barra "
            "lateral esquerda > Add new secret > nome 'OPENAI_API_KEY', "
            "valor a chave da API — e habilite o acesso do notebook a esse "
            "secret. Depois rode esta célula de novo."
        )
    _os.environ["OPENAI_API_KEY"] = chave
    print("✅ Chave da OpenAI carregada — nenhum Qwen local será carregado neste modo.")

In [ ]:
# ============================================================
# Célula 3: Carregar grafo do módulo graph/agent.py
# ============================================================
import sys

# Limpa cache do Python para forçar reimport do disco
mods_remover = [k for k in sys.modules if "graph" in k or "schemas" in k or "agent" in k]
for mod in mods_remover:
    del sys.modules[mod]

# Importa do zero
import graph.agent as agent_module

print(f"campo_prioritario no prompt: {'campo_prioritario' in agent_module.PROMPT_PERGUNTA}")
print(f"Arquivo: {agent_module.__file__}")

from graph.agent import inicializar_modelo, construir_grafo, processar_turno
from schemas.models import ModoExecucao

# Bloco 15: modo agora é obrigatório — construído a partir da MODO_EXECUCAO
# (string) definida na Célula 1. Antes essa chamada não passava o 3º
# argumento e caía silenciosamente em GPU_LOCAL, mesmo com CPU_LOCAL
# selecionado — foi exatamente isso que causou o "Modo de execução ativo:
# GPU_LOCAL" com MODO_EXECUCAO="CPU_LOCAL".
inicializar_modelo(model, tokenizer, ModoExecucao(MODO_EXECUCAO))
agente = construir_grafo()
print(f"✅ Grafo compilado — nós: {list(agente.get_graph().nodes.keys())}")

# **Interface**

In [ ]:
# ============================================================
# DataBrief AI — Ato 1
# Célula 4: Lançar interface Gradio
# ============================================================
import sys
for mod in list(sys.modules.keys()):
    if "interface" in mod:
        del sys.modules[mod]

from interface.app import construir_interface

demo = construir_interface(agente)
demo.launch(
    share=True,
    debug=False,
    show_error=True,
)

In [ ]:
   import glob, os
   achados = glob.glob('/content/**/testes/conftest.py', recursive=True)
   pasta = os.path.dirname(os.path.dirname(achados[0]))
   print('Projeto encontrado em:', pasta)
   !cd "{pasta}" && python -m pytest testes -q